# Lab Day 10 — E0–E4
Bản làm việc tiếp tục ngày 14/10/2026. E0 test và E1/E2 train + val đã xong; E3 smoke scale 1024 đạt 8 bước, **full E3 chưa chạy**. Chạy mục 0–2 để khôi phục, rồi mục 5–6 với EXP=e3. Không Run all; không nạp model trên local.

`lab_day10_v1.ipynb` là bản tải về giữ nguyên lịch sử. Phụ lục cuối notebook trích log thật từ bản đó; các cell mới chưa chạy không có output giả.

## 0. Môi trường và upload source
Windows: `python -m src.package_colab`. Mở notebook này trên Colab, chọn T4 GPU, upload `work/colab_bundle.zip` ở cell đầu. Không clone origin vì code local chưa push. Bỏ qua cell upload nếu workspace đã tồn tại.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files
ROOT = Path("/content/vlm_lab")
if ROOT.exists():
    raise FileExistsError("Workspace đã có; bỏ qua upload, chạy cell thiết lập tiếp theo.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Chọn đúng một file colab_bundle.zip")
with zipfile.ZipFile(next(iter(uploaded))) as bundle:
    for member in bundle.infolist():
        if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError("Đường dẫn ZIP không an toàn")
    bundle.extractall(ROOT)
print(ROOT)

In [ ]:
%cd /content/vlm_lab
import os
import sys
import subprocess
from pathlib import Path
from src.colab_workflow import run
run("-m", "pip", "install", "-r", "requirements-colab.txt")
run("-m", "pip", "install", "pytest")
# torchao là backend tùy chọn không dùng trong bài; bản Colab 0.10.0 đã gây lỗi PEFT.
import importlib.metadata as metadata
from packaging.version import Version
try:
    torchao_version = metadata.version("torchao")
except metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and Version(torchao_version) < Version("0.16.0"):
    print("Gỡ torchao không tương thích:", torchao_version)
    run("-m", "pip", "uninstall", "-y", "torchao")
# Nếu cần restart session, chạy lại cell này, bỏ qua upload source.

In [ ]:
import importlib.metadata as metadata
import torch
assert torch.cuda.is_available(), "Chọn runtime T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
for package in ("transformers", "peft", "accelerate", "bitsandbytes", "trl", "torchvision"):
    print(package, metadata.version(package))
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/day10_vlm")
BACKUP = str(DRIVE_ROOT / "runs")
Path(BACKUP).mkdir(parents=True, exist_ok=True)
RUN_TAGS = {"e1": "e1", "e2": "e2", "e3": "e3_scale1024"}

### Khôi phục sau nghỉ GPU
Chỉ chạy cell sau khi mount Drive. Khôi phục E0, val E1/E2, checkpoint đầy đủ E1/E2 và smoke E3 scale1024. Không ghi đè source ZIP mới bằng backup cũ. Checkpoint có thể mất vài phút để chép; không nạp model ở bước này.

In [ ]:
from src.colab_workflow import restore_from_drive
RESTORE_RUNS = ["e1", "e2", "e3_scale1024_smoke"]
if (DRIVE_ROOT / "runs/e3_scale1024/step-0300/complete.json").exists():
    RESTORE_RUNS.append("e3_scale1024")
restore_from_drive(DRIVE_ROOT, RESTORE_RUNS)
for name in ("report/evidence/e0.json", "work/validation/e1.json", "work/validation/e2.json",
             "work/runs/e1/step-0300/complete.json", "work/runs/e2/step-0300/complete.json",
             "work/runs/e3_scale1024_smoke/smoke_pass.json"):
    assert Path(name).exists(), name
    print("OK", name)

## 1. Dữ liệu và split chuẩn
Thêm KAGGLE_USERNAME/KAGGLE_KEY vào Colab Secrets, bật notebook access. Không hiển thị giá trị. Nếu dùng dataset ZIP trong Drive, thay lệnh --kaggle bằng --zip đường dẫn ZIP.

In [ ]:
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
if Path("data/splits/meta.json").exists():
    print("Giữ split đã có")
else:
    run("scripts/prepare_data.py", "--kaggle")

In [ ]:
from src.data import inspect_data, read_split
from src.visualize import draw_examples
from IPython.display import display, Image
meta = inspect_data("data")
for path in draw_examples(out="work/label_preview"):
    display(Image(filename=str(path)))

## 2. Nhãn YOLO → JSON 0–1000
Kiểm tra thứ tự box và box chuyển ngược về pixel ở mục 1. Chạy test gốc liên quan tới giao diện chấm và cấu trúc notebook.

In [ ]:
from src.coordinates import objects_to_answer
row = read_split("data", "train")[0]
print(row["image_id"], objects_to_answer(row["objects"], row["width"], row["height"]))
print("Ảnh rỗng:", objects_to_answer([]))
# Các test này đã đạt ở local. Chỉ chạy lại khi có thay đổi liên quan:
# run("-m", "pytest", "-q", "tests/test_score.py", "tests/test_configs_and_docs.py")

## 3. E0 — smoke rồi mới test
Cell đầu chỉ 2 ảnh val. Quan sát output thô; gửi lỗi/log nếu có. Sau khi đường chạy ổn mới chạy cell toàn bộ test.

In [ ]:
if Path("report/evidence/e0.json").exists():
    print("E0 đã hoàn thành; bỏ qua smoke. Xem output smoke thật trong phụ lục.")
else:
    run("-m", "src.run", "predict", "--config", "configs/e0.yaml", "--split", "val",
        "--limit", "2", "--batch-size", "2", "--out", "work/e0_smoke_fixed.jsonl")
    print(Path("work/e0_smoke_fixed.jsonl").read_text())

In [ ]:
import json
if Path("report/evidence/e0.json").exists():
    e0 = json.loads(Path("report/evidence/e0.json").read_text())
    assert e0["dataset_version"] == meta["dataset_version"]
    assert e0["split"] == "test" and e0["n_images"] == 311 and e0["missing_predictions"] == 0
    print("Giữ nguyên E0 đã hoàn thành:", e0["micro"], e0["f1_ci95"])
else:
    run("-m", "src.run", "predict", "--config", "configs/e0.yaml", "--split", "test",
        "--out", "work/predictions/e0_test.jsonl")
    run("-m", "src.run", "score", "--pred", "work/predictions/e0_test.jsonl", "--split", "test",
        "--name", "E0", "--out", "report/evidence/e0.json")

## 4. Dữ liệu SFT và mask loss
Module src.sft kiểm tra prompt là tiền tố, mask prompt/ảnh/padding, giữ JSON + token kết thúc. Smoke mục 6 in decode labels và kiểm tra 220 token ảnh 640×360. Không cắt input âm thầm.

## 5. Chọn setup — hiện tiếp tục E3
E1/E2 đã train và đánh giá val; E3 có 348 module LoRA, 148.426.752 tham số trainable, projector 109.105.152. Dùng scale 1024 sau lỗi gradient ở scale mặc định. Đây là thay đổi ổn định số học phải ghi nhận khi báo cáo; chưa kết luận về chất lượng E3. E1–E3 đều khởi tạo độc lập từ model gốc.

In [ ]:
import json
from src.config import load_config, fingerprint
from src.colab_workflow import training_overrides
EXP = "e3"  # Hiện tiếp tục E3; E1/E2 đã hoàn thành.
TAG = RUN_TAGS[EXP]
CFG = f"configs/{EXP}.yaml"
EFFECTIVE_CONFIG = load_config(CFG)
if EXP == "e3":
    EFFECTIVE_CONFIG["amp_init_scale"] = 1024.0
TRAIN_ARGS = training_overrides(EFFECTIVE_CONFIG)
SMOKE_STEPS = 8 if EXP == "e3" else 2
SMOKE = f"work/runs/{TAG}_smoke"
OUT = f"work/runs/{TAG}"
CHECKPOINT = f"{OUT}/step-0300"
print(EXP, TAG, SMOKE, CHECKPOINT, TRAIN_ARGS)

## 6. Smoke và full train — giữ scale khớp nhau
E3 dùng batch 2/accum 4, LR gốc, AMP init_scale=1024; smoke 8 bước đã đạt và được khôi phục từ Drive. Cell smoke bên dưới chỉ kiểm tra fingerprint và đọc marker nếu đã có. **Cell tiếp theo là full train 300 bước**, cần ngân sách GPU đủ dài (E1/E2 thực đo khoảng 45 phút, E3 có thể lâu hơn). Không chạy full khi chỉ còn khoảng 40 phút. Checkpoint hiện chưa hỗ trợ resume optimizer/RNG chính xác.

In [ ]:
marker_path = Path(SMOKE) / "smoke_pass.json"
if marker_path.exists():
    marker = json.loads(marker_path.read_text())
    assert marker["fingerprint"] == fingerprint(EFFECTIVE_CONFIG, meta["dataset_version"])
    print("Tái sử dụng smoke đã đạt:", marker)
else:
    run("-m", "src.run", "train", "--config", CFG, "--out", SMOKE,
        "--smoke", "--smoke-steps", str(SMOKE_STEPS), "--backup", BACKUP, *TRAIN_ARGS)
    print(marker_path.read_text())

In [ ]:
# Đây là cell FULL TRAIN duy nhất cho EXP hiện tại. Cần đủ thời gian GPU.
completed = Path(CHECKPOINT) / "complete.json"
if completed.exists():
    saved_run = json.loads((Path(CHECKPOINT) / "run.json").read_text())
    assert not saved_run["smoke"] and saved_run["steps"] == 300
    assert saved_run["fingerprint"] == fingerprint(EFFECTIVE_CONFIG, meta["dataset_version"])
    print("Run đã hoàn thành; không train lại.")
else:
    run("-m", "src.run", "train", "--config", CFG, "--out", OUT,
        "--smoke-dir", SMOKE, "--backup", BACKUP, *TRAIN_ARGS)
print(Path(f"{OUT}/run.json").read_text())

Đánh giá toàn bộ val để so sánh setup. Chưa dùng test để gỡ lỗi. Gửi setup.json, run.json, train_log.jsonl và kết quả val. Checkpoint/log tự sao lưu Drive mỗi 100 bước; chạy cell sao lưu cuối notebook sau mỗi setup để giữ thêm evidence/prediction.

In [ ]:
val_score = Path(f"work/validation/{EXP}.json")
if val_score.exists():
    result = json.loads(val_score.read_text())
    assert result["split"] == "val" and result["missing_predictions"] == 0
    assert result["dataset_version"] == meta["dataset_version"]
    print("Giữ kết quả val đã có:", result)
else:
    run("-m", "src.run", "predict", "--checkpoint", CHECKPOINT, "--split", "val",
        "--out", f"work/predictions/{TAG}_val.jsonl")
    run("-m", "src.run", "score", "--checkpoint", CHECKPOINT, "--split", "val",
        "--pred", f"work/predictions/{TAG}_val.jsonl", "--name", EXP.upper(),
        "--out", str(val_score))
    print(val_score.read_text())

## 7. Test và evidence E1–E3
Chỉ khi cả ba setup đã qua smoke/full/val. Mỗi lệnh GPU chạy process riêng để giải phóng VRAM. Chấm test một lần/setup; meta lấy từ checkpoint thật.

In [ ]:
for exp in ("e1", "e2", "e3"):
    tag = RUN_TAGS[exp]
    checkpoint = f"work/runs/{tag}/step-0300"
    pred = f"work/predictions/{tag}_test.jsonl"
    run("-m", "src.run", "predict", "--checkpoint", checkpoint, "--split", "test", "--out", pred)
    run("-m", "src.run", "score", "--checkpoint", checkpoint, "--split", "test",
        "--pred", pred, "--name", exp.upper(), "--out", f"report/evidence/{exp}.json")

## 8. E4 — data_size 10% / 100%
Chọn setup bằng F1 val E1–E3 (hòa ưu tiên setup đơn giản). Đổi đúng train_fraction, vẫn 300 bước, các thông số khác giữ nguyên đối chứng. 10% train từ model gốc; 100% tái sử dụng đúng run/prediction của đối chứng. Chưa làm E5.

In [ ]:
import json
from src.results import choose_by_val
from src.colab_workflow import e4_overrides
BASE = choose_by_val()
BASE_TAG = RUN_TAGS[BASE]
BASE_CHECKPOINT = f"work/runs/{BASE_TAG}/step-0300"
base_config = json.loads(Path(f"{BASE_CHECKPOINT}/config.json").read_text())
# Kế thừa cả batch/accum và AMP scale; chỉ thay fraction.
E4_TRAIN_ARGS = e4_overrides(BASE_CHECKPOINT, f"configs/{BASE}.yaml")
E4_TAG = f"e4_data_size_0.1_{BASE_TAG}"
E4_SMOKE = f"work/runs/{E4_TAG}_smoke"
E4_OUT = f"work/runs/{E4_TAG}"
E4_CHECKPOINT = f"{E4_OUT}/step-0300"
print("Đối chứng:", BASE, "Overrides:", E4_TRAIN_ARGS)

In [ ]:
run("-m", "src.run", "train", "--config", f"configs/{BASE}.yaml", "--fraction", "0.1",
    "--out", E4_SMOKE, "--smoke", "--smoke-steps", "8", "--backup", BACKUP, *E4_TRAIN_ARGS)

In [ ]:
run("-m", "src.run", "train", "--config", f"configs/{BASE}.yaml", "--fraction", "0.1",
    "--out", E4_OUT, "--smoke-dir", E4_SMOKE, "--backup", BACKUP, *E4_TRAIN_ARGS)

In [ ]:
run("-m", "src.run", "predict", "--checkpoint", E4_CHECKPOINT, "--split", "val",
    "--out", f"work/predictions/{E4_TAG}_val.jsonl")
run("-m", "src.run", "score", "--checkpoint", E4_CHECKPOINT, "--split", "val",
    "--pred", f"work/predictions/{E4_TAG}_val.jsonl", "--name", "E4_data_size_0.1",
    "--out", "work/validation/e4_data_size_0.1.json")

Sau khi val E4 ổn mới chạy test và tạo cả hai evidence. Không cần train/suy luận lại đối chứng 100%.

In [ ]:
run("-m", "src.run", "predict", "--checkpoint", E4_CHECKPOINT, "--split", "test",
    "--out", f"work/predictions/{E4_TAG}_test.jsonl")
run("-m", "src.run", "score", "--checkpoint", E4_CHECKPOINT, "--split", "test",
    "--pred", f"work/predictions/{E4_TAG}_test.jsonl", "--name", "E4_data_size_0.1",
    "--out", "report/evidence/e4_data_size_0.1.json")
run("-m", "src.run", "score", "--checkpoint", BASE_CHECKPOINT, "--split", "test",
    "--pred", f"work/predictions/{BASE_TAG}_test.jsonl", "--name", "E4_data_size_1.0",
    "--out", "report/evidence/e4_data_size_1.0.json")

### Bảng và hình minh họa từ kết quả thật
6 ảnh đầu chỉ để bắt đầu xem lỗi. Khi viết REPORT cần chọn cả ca đúng/sai và tự chú thích cụ thể. GT xanh, PRED đỏ. REPORT chưa được tự điền.

In [ ]:
from src.results import results_table
for result in results_table():
    print(result)
for path in draw_examples(split="test", prediction=f"work/predictions/{BASE_TAG}_test.jsonl"):
    display(Image(filename=str(path)))

### Sao lưu artifacts — chạy sau mỗi setup
Checkpoint/log train đã tự backup; cell này giữ thêm evidence, hình, predictions, validation và source. Lưu notebook có output bằng File → Save a copy in Drive; file notebook trên disk chưa chứa output UI.

In [ ]:
import shutil
destination = Path("/content/drive/MyDrive/day10_vlm/artifacts")
destination.mkdir(parents=True, exist_ok=True)
for name in ("report", "work/predictions", "work/validation", "work/logs", "src", "configs"):
    if Path(name).exists():
        shutil.copytree(name, destination / name, dirs_exist_ok=True)
for name in ("REPORT.md", "lab_day10.ipynb", "requirements-colab.txt", "COLAB_RUN_GUIDE.md"):
    shutil.copy2(name, destination / name)
shutil.copy2("data/splits/meta.json", destination / "dataset_meta.json")
print(destination)

### Trước khi nộp
Chỉ sau khi có số thật, REPORT hoàn chỉnh, ≥6 ảnh có chú thích và ≥5 commit: chạy toàn bộ tests + checker. Chạy hai cell riêng để vẫn thấy checker nếu pytest lỗi. ZIP không có .git; kiểm tra tracked files/commit cần chạy lại trong repo Git local. Xem ghi chú test REPORT yêu cầu marker trái với checker trong COLAB_RUN_GUIDE.

In [ ]:
# Chỉ khi chuẩn bị nộp:
# run("-m", "pytest", "-q")

In [ ]:
# run("scripts/check_submission.py")

### Nhật ký đã chạy — trích từ lab_day10_v1.ipynb
Các đoạn dưới là output thật từ notebook người dùng tải về ngày 14/10/2026, **không phải output chạy lại của các cell đã chỉnh sửa**. File v1 giữ nguyên cả cell lỗi và ảnh. E0 là test, E1/E2 là val; không so trực tiếp F1 giữa hai split. Full E3, test E1–E3 và E4 chưa có kết quả.

Lịch sử sửa lỗi: skip quantization đổi thành model.visual; gỡ torchao 0.10.0 không tương thích; E3 gradient không hữu hạn ở bước 6 tại scale mặc định, batch 1/accum 8 vẫn lỗi bước 1; smoke scale1024 đạt 8 bước. Chưa khẳng định scale1024 sẽ ổn trên toàn bộ train.

Trích output cell 4 (chỉ số bắt đầu từ 0) trong bản v1:

```text
GPU: Tesla T4 torch: 2.11.0+cu130
transformers 5.19.0
peft 0.21.2
accelerate 1.15.0
bitsandbytes 0.50.2
trl 1.9.2
torchvision 0.26.0+cu130
Mounted at /content/drive
```

Trích output cell 7 (chỉ số bắt đầu từ 0) trong bản v1:

```text
{
  "dataset_version": "d10-21e4070301",
  "counts": {
    "train": {
      "images": 1422,
      "boxes": {
        "manhole": 691,
        "pothole": 1074,
        "crack": 1750
      }
    },
    "val": {
      "images": 276,
      "boxes": {
        "crack": 470,
        "manhole": 103,
        "pothole": 66
      }
    },
    "test": {
      "images": 311,
      "boxes": {
        "crack": 299,
        "manhole": 163,
        "pothole": 121
      }
    }
  }
}
```

Trích output cell 14 (chỉ số bắt đầu từ 0) trong bản v1:

```text
{"image_id": "vlcsnap-2025-02-18-18h30m17s408", "raw": "[{\"label\": \"crack\", \"bbox_2d\": [0, 370, 1000, 700]}, {\"label\": \"crack\", \"bbox_2d\": [0, 370, 1000, 700]}, {\"label\": \"crack\", \"bbox_2d\": [0, 370, 1000, 700]}, {\"label\": \"crack\", \"bbox_2d\": [0, 370, 1000, 700]}, {\"label\": \"crack\", \"bbox_2d\": [0, 370, 1000, 700]}, {\"label\": \"crack\", \"bbox_2d\": [0, 370, 1000, 700]}, {\"label\": \"crack\", \"bbox_2d\": [0, 370, 1000, 700]}, {\"label\": \"crack\", \"bbox_2d\": [0, 370, 1000, 700"}
{"image_id": "vlcsnap-2025-02-18-18h30m20s293", "raw": "[{\"label\": \"crack\", \"bbox_2d\": [0, 500, 1000, 700]}, {\"label\": \"crack\", \"bbox_2d\": [0, 400, 1000, 600]}, {\"label\": \"crack\", \"bbox_2d\": [0, 300, 1000, 500]}, {\"label\": \"crack\", \"bbox_2d\": [0, 200, 1000, 400]}, {\"label\": \"crack\", \"bbox_2d\": [0, 100, 1000, 300]}, {\"label\": \"crack\", \"bbox_2d\": [0, 0, 1000, 200]}, {\"label\": \"manhole\", \"bbox_2d\": [917, 500, 953, 590]}]"}
```

Trích output cell 36 (chỉ số bắt đầu từ 0) trong bản v1:

```text
{
  "n_images": 276,
  "parse_valid_rate": 0.9927536231884058,
  "missing_predictions": 0,
  "micro": {
    "precision": 0.22445255474452555,
    "recall": 0.19248826291079812,
    "f1": 0.20724515585509692,
    "tp": 123,
    "fp": 425,
    "fn": 516
  },
  "per_class": {
    "pothole": {
      "precision": 0.140625,
      "recall": 0.13636363636363635,
      "f1": 0.13846153846153844,
      "tp": 9,
      "fp": 55,
      "fn": 57
    },
    "crack": {
      "precision": 0.15550239234449761,
      "recall": 0.13829787234042554,
      "f1": 0.1463963963963964,
      "tp": 65,
      "fp": 353,
      "fn": 405
    },
    "manhole": {
      "precision": 0.7424242424242424,
      "recall": 0.47572815533980584,
      "f1": 0.5798816568047338,
      "tp": 49,
      "fp": 17,
      "fn": 54
    },
    "other": {
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "tp": 0,
      "fp": 0,
      "fn": 0
    }
  },
  "mean_iou": 0.6870305872483309,
  "fp_per_image": 1.539855072463768,
  "f1_ci95": [
    0.17264957264957265,
    0.24126455906821964
  ],
  "score_version": 1,
  "name": "E1",
  "dataset_version": "d10-21e4070301",
  "split": "val",
  "coord": "norm1000",
  "meta": {
    "trainable_params": 33030144,
    "peak_vram_gb": 6.428408145904541,
    "train_minutes": 44.68038807185,
    "base_setup": "E1",
    "train_fraction": 1.0
  }
}
```

Trích output cell 37 (chỉ số bắt đầu từ 0) trong bản v1:

```text
{
  "n_images": 276,
  "parse_valid_rate": 1.0,
  "missing_predictions": 0,
  "micro": {
    "precision": 0.33206831119544594,
    "recall": 0.27386541471048514,
    "f1": 0.30017152658662094,
    "tp": 175,
    "fp": 352,
    "fn": 464
  },
  "per_class": {
    "pothole": {
      "precision": 0.13333333333333333,
      "recall": 0.24242424242424243,
      "f1": 0.17204301075268819,
      "tp": 16,
      "fp": 104,
      "fn": 50
    },
    "crack": {
      "precision": 0.3188854489164087,
      "recall": 0.21914893617021278,
      "f1": 0.2597730138713745,
      "tp": 103,
      "fp": 220,
      "fn": 367
    },
    "manhole": {
      "precision": 0.6666666666666666,
      "recall": 0.5436893203883495,
      "f1": 0.5989304812834224,
      "tp": 56,
      "fp": 28,
      "fn": 47
    },
    "other": {
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "tp": 0,
      "fp": 0,
      "fn": 0
    }
  },
  "mean_iou": 0.673363858201567,
  "fp_per_image": 1.2753623188405796,
  "f1_ci95": [
    0.259529602595296,
    0.3408476104598737
  ],
  "score_version": 1,
  "name": "E2",
  "dataset_version": "d10-21e4070301",
  "split": "val",
  "coord": "norm1000",
  "meta": {
    "trainable_params": 142135296,
    "peak_vram_gb": 8.122631549835205,
    "train_minutes": 45.796418932133356,
    "base_setup": "E2",
    "train_fraction": 1.0
  }
}
```

Trích output cell 31 (chỉ số bắt đầu từ 0) trong bản v1; đã bỏ dòng tiến độ Loading weights, giữ log còn lại:

```text
python -m src.run train --config configs/e3.yaml --out work/runs/e3_scale1024_smoke --smoke --smoke-steps 8 --backup /content/drive/MyDrive/day10_vlm/runs --amp-init-scale 1024

Quantization OK: LLM 4-bit; visual và lm_head không bị lượng tử hóa.
{'matched_modules': 348, 'trainable_params': 148426752, 'projector_params': 109105152}
{'image_tokens': 220, 'supervised_tokens': 250, 'decoded_labels': '[{"label":"crack","bbox_2d":[430,467,583,506]},{"label":"pothole","bbox_2d":[361,475,414,522]},{"label":"manhole","bbox_2d":[317,481,616,625]},{"label":"pothole","bbox_2d":[583,500,627,614]},{"label":"crack","bbox_2d":[580,508,655,614]},{"label":"pothole","bbox_2d":[259,517,311,572]},{"label":"crack","bbox_2d":[397,628,447,717]},{"label":"crack","bbox_2d":[339,683,398,808]},{"label":"crack","bbox_2d":[158,789,361,994]}]<|im_end|>\n'}
{'amp_init_scale': 1024.0}
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'step': 1, 'loss': 1.1938429474830627, 'grad_norm': 5.751036167144775, 'lr': 2.6666666666666667e-05, 'amp_scale': 1024.0}
{'step': 8, 'loss': 0.5885869711637497, 'grad_norm': 1.7094208002090454, 'lr': 0.00012, 'amp_scale': 1024.0}
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Quantization OK: LLM 4-bit; visual và lm_head không bị lượng tử hóa.
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
predict 2/2
Smoke output: [{'image_id': 'vlcsnap-2025-02-18-18h30m17s408', 'raw': '[]'}, {'image_id': 'vlcsnap-2025-02-18-18h30m20s293', 'raw': '[{"label":"crack","bbox_2d":[10,550,197,650]}]'}]
SMOKE PASS — có thể chạy full với --smoke-dir work/runs/e3_scale1024_smoke
Checkpoint: work/runs/e3_scale1024_smoke/step-0008
{
  "fingerprint": "95d3baa1b385010194dd19c60d1f6f48c79829b0dc85737565b9ca89882d7efc",
  "steps": 8,
  "adapter_roundtrip": true,
  "projector_roundtrip": true
}
```

Trích output cell 51 (chỉ số bắt đầu từ 0) trong bản v1:

```text
runs/e1/step-0300/complete.json OK
runs/e2/step-0300/complete.json OK
runs/e3_scale1024_smoke/smoke_pass.json OK
artifacts/src/train.py OK
artifacts/src/run.py OK
artifacts/work/validation/e1.json OK
artifacts/work/validation/e2.json OK
artifacts/report/evidence/e0.json OK
```

Trích output cell 52 (chỉ số bắt đầu từ 0) trong bản v1:

```text
work/predictions/e0_test.jsonl OK
report/evidence/e0.json OK
{
  "n_images": 311,
  "parse_valid_rate": 0.40192926045016075,
  "missing_predictions": 0,
  "micro": {
    "precision": 0.06666666666666667,
    "recall": 0.0411663807890223,
    "f1": 0.05090137857900318,
    "tp": 24,
    "fp": 336,
    "fn": 559
  },
  "per_class": {
    "pothole": {
      "precision": 0.2222222222222222,
      "recall": 0.06611570247933884,
      "f1": 0.10191082802547771,
      "tp": 8,
      "fp": 28,
      "fn": 113
    },
    "crack": {
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "tp": 0,
      "fp": 189,
      "fn": 299
    },
    "manhole": {
      "precision": 0.11851851851851852,
      "recall": 0.09815950920245399,
      "f1": 0.10738255033557048,
      "tp": 16,
      "fp": 119,
      "fn": 147
    },
    "other": {
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "tp": 0,
      "fp": 0,
      "fn": 0
    }
  },
  "mean_iou": 0.5978109380264378,
  "fp_per_image": 1.0803858520900322,
  "f1_ci95": [
    0.03289473684210526,
    0.07127882599580714
  ],
  "score_version": 1,
  "name": "E0",
  "dataset_version": "d10-21e4070301",
  "split": "test",
  "coord": "norm1000",
  "meta": {}
}
```